<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [ ]:
# Implemente uma função load_data(seed) que carregue o dataset,
# realize a separação em treino e teste com controle de aleatoriedade
# e retorne X_train, X_test, y_train, y_test.

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed=42):
    dataset = fetch_openml("mnist_784", version=1, as_frame=False)
    X = dataset.data
    y = dataset.target.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        stratify=y,
        random_state=seed
    )

    return X_train, X_test, y_train, y_test


# Teste rápido
X_train, X_test, y_train, y_test = load_data(seed=42)
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

**É necessário normalizar os dados para esse tipo de modelo?**

Não é estritamente necessário para Random Forest e AdaBoost. Ambos são modelos baseados em árvores de decisão, tentando descobrir se o valor é menor ou maior, o que é invariante à escala.

Ainda assim, normalizar (dividindo por 255) pode acelerar levemente o treinamento ao reduzir a busca por limiares em alguns casos, mas não altera a acurácia do modelo.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
# Implemente train_random_forest e train_adaboost,
# garantindo reprodutibilidade com random_state=seed.

from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

def train_random_forest(X_train, y_train, seed=42):
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    return model


def train_adaboost(X_train, y_train, seed=42):
    model = AdaBoostClassifier(
        n_estimators=100,
        random_state=seed
    )
    model.fit(X_train, y_train)
    return model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [ ]:
# Implemente evaluate(model, X_test, y_test):
# realiza predições e retorna a acurácia do modelo.

from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return acc


print("Função evaluate definida.")

A função evaluate recebe o modelo já treinado, realiza as predições no conjunto de teste e retorna a acurácia.

Manter a função separada do treinamento é uma boa prática: permite reutilizá-la com qualquer modelo e facilita análises.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [ ]:
# Implemente run_pipeline(model_type, seed):
# carrega dados, treina o modelo escolhido (rf ou ab), avalia e retorna a acurácia.

def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed=seed)

    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed=seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed=seed)
    else:
        raise ValueError(f"model_type inválido: '{model_type}'. Use 'rf' ou 'ab'.")

    acc = evaluate(model, X_test, y_test)
    return acc


# Teste básico
acc_rf = run_pipeline(model_type="rf", seed=42)
print(f"Acurácia Random Forest (seed=42): {acc_rf:.4f}")

O run_pipeline encapsula todo o fluxo do experimento em uma única chamada. Isso é fundamental para reprodutibilidade: ao passar um seed fixo, qualquer pessoa que rode a mesma lógica obtém exatamente os mesmos resultados.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [ ]:
# Execute o pipeline para Random Forest e AdaBoost.
# Apresente acurácia, precisão, recall e F1-Score de cada modelo.

from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

SEED = 42
X_train, X_test, y_train, y_test = load_data(seed=SEED)

rf_model = train_random_forest(X_train, y_train, seed=SEED)
ab_model = train_adaboost(X_train, y_train, seed=SEED)

for name, model in [("Random Forest", rf_model), ("AdaBoost", ab_model)]:
    y_pred = model.predict(X_test)
    print(f"{'='*50}")
    print(f"Modelo: {name}")
    print(f"  Acurácia  : {accuracy_score(y_test, y_pred):.4f}")
    print(f"  Precisão  : {precision_score(y_test, y_pred, average='weighted'):.4f}")
    print(f"  Recall    : {recall_score(y_test, y_pred, average='weighted'):.4f}")
    print(f"  F1-Score  : {f1_score(y_test, y_pred, average='weighted'):.4f}")
    print()
    print(classification_report(y_test, y_pred))

**Qual modelo apresentou melhor desempenho inicial?**

O Random Forest apresenta desempenho inicial significativamente superior ao AdaBoost neste dataset. Random Forest combina centenas de árvores profundas de forma paralela, o que o torna naturalmente robusto em datasets de alta dimensionalidade. O AdaBoost, por sua vez, usa árvores rasas, o que o torna mais lento para convergir convergir e menos eficaz.

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [ ]:
# Execute o pipeline com diferentes seeds e analise se os resultados mudam.

import pandas as pd

seeds = [42, 7]
records = []

for seed in seeds:
    for model_type, name in [("rf", "Random Forest"), ("ab", "AdaBoost")]:
        acc = run_pipeline(model_type=model_type, seed=seed)
        records.append({"seed": seed, "modelo": name, "acurácia": round(acc, 4)})

df = pd.DataFrame(records)
print(df.to_string(index=False))

**Os resultados mudaram? O experimento é reprodutível?**

Os resultados variam levemente entre seeds diferentes, isso é o esperado: seeds diferentes geram divisões de treino/teste diferentes. O que garante reprodutibilidade não é obter o mesmo número com qualquer seed, mas sim obter **sempre o mesmo número para a mesma seed**.

O experimento é reprodutível. O random_state está fixado em todas as etapas, o que elimina qualquer fonte de aleatoriedade não controlada.

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [ ]:
# Compare acurácia em treino e teste para ambos os modelos.
# Analise a existência de overfitting.

SEED = 42
X_train, X_test, y_train, y_test = load_data(seed=SEED)

rf_model = train_random_forest(X_train, y_train, seed=SEED)
ab_model = train_adaboost(X_train, y_train, seed=SEED)

for name, model in [("Random Forest", rf_model), ("AdaBoost", ab_model)]:
    acc_treino = evaluate(model, X_train, y_train)
    acc_teste  = evaluate(model, X_test,  y_test)
    gap = acc_treino - acc_teste
    print(f"{name}")
    print(f"  Acurácia treino : {acc_treino:.4f}")
    print(f"  Acurácia teste  : {acc_teste:.4f}")
    print(f"  Gap (overfitting): {gap:.4f}")
    print()

**Existe overfitting? Qual modelo sofre mais?**

O Random Forest apresenta overfitting claro: acurácia próxima de 100% no treino e menor no teste. Isso acontece porque cada árvore individual cresce sem restrição de profundidade, memorizando os dados de treino.

O AdaBoost tende a ter um gap de treino e teste menor. Por usar árvores rasas (depth=1 por padrão), cada estimador individual é fraco demais para memorizar. Portanto, o Random Forest tende a possuir maior overfitting neste cenário.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [ ]:
# Varie n_estimators em Random Forest e AdaBoost.
# Analise o impacto no desempenho.

import pandas as pd
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score

SEED = 42
X_train, X_test, y_train, y_test = load_data(seed=SEED)

n_estimators_list = [10, 50, 100, 200]
records = []

for n in n_estimators_list:
    # Random Forest
    rf = RandomForestClassifier(n_estimators=n, random_state=SEED, n_jobs=-1)
    rf.fit(X_train, y_train)
    acc_rf = accuracy_score(y_test, rf.predict(X_test))

    # AdaBoost
    ab = AdaBoostClassifier(n_estimators=n, random_state=SEED)
    ab.fit(X_train, y_train)
    acc_ab = accuracy_score(y_test, ab.predict(X_test))

    records.append({
        "n_estimators": n,
        "acc_rf":        round(acc_rf, 4),
        "acc_ab":        round(acc_ab, 4)
    })

df = pd.DataFrame(records)
print(df.to_string(index=False))

**O desempenho muda significativamente? Qual modelo é mais sensível?**

O Random Forest satura rapidamente: a acurácia melhora de 10 para 50 estimadores, mas de 100 em diante o ganho é marginal. Adicionar mais árvores reduz a variância do ensemble, mas após certo ponto o retorno diminui. O custo computacional cresce linearmente enquanto o benefício em acurácia praticamente desaparece.

O AdaBoost é mais sensível ao n_estimators: com poucos estimadores (10) o desempenho é fraco porque o algoritmo não teve iterações suficientes para corrigir todos os erros. O desempenho cresce mais consistentemente com o aumento de estimadores. Portanto, o AdaBoost é mais sensível a essa mudança.

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

**1. A acurácia é suficiente para avaliar os modelos?**

Não necessariamente. A acurácia mede apenas a proporção de acertos totais, o que pode ser enganoso quando as classes estão desbalanceadas.

Para uma avaliação mais robusta, precisão, recall e F1-score por classe são essenciais.

**2. Como você garante que o resultado não ocorreu por acaso?**

A principal garantia é o uso de random_state fixo em todas as etapas: divisão treino/teste, inicialização dos modelos e seleção de features. Isso torna o experimento determinístico, o mesmo resultado é obtido toda vez que o código é executado com a mesma seed.

Para garantia estatística mais rigorosa, o ideal seria usar validação cruzada (k-fold), que avalia o modelo em múltiplas partições diferentes dos dados. Isso reduziria a influência da divisão específica escolhida e daria um intervalo de confiança para a acurácia, permitindo afirmar com mais segurança que o desempenho é estável e não resultado de uma partição favorável.

**3. Cite dois possíveis problemas metodológicos neste experimento.**

Primeiro, o uso de uma única divisão treino/teste é um problema. A acurácia obtida depende de quais exemplos caíram em cada partição: uma divisão diferente poderia gerar resultados ligeiramente diferentes. Isso torna a comparação entre modelos menos confiável do que uma avaliação por validação cruzada.

Segundo, os hiperparâmetros padrão foram usados sem busca sistemática como no GridSearch. Isso significa que os modelos podem não estar em seu melhor estado para comparação. Comparar Random Forest e AdaBoost com parâmetros arbitrários pode favorecer injustamente determinado modelo.

**4. O pipeline implementado é confiável? Justifique.**

O pipeline é confiável dentro do escopo proposto. Ele garante reprodutibilidade (todas as fontes de aleatoriedade estão controladas via seed), é modular, e segue boas práticas de separação treino/teste com estratificação.